# Global Environmental Intelligence — Business Analysis
### *5 Questions Every Environmental Policymaker Should Be Able to Answer*

**Author:** Senior Data & Business Analyst | **Dataset:** Global Environmental Intelligence Dataset  
**Tools:** Python · Pandas · Plotly

---
> **Executive Summary:**  
> Air pollution is the world's largest environmental health risk, killing **7 million people per year**.  
> This notebook answers five business-critical questions using WHO, IHME, and OpenAQ data —  
> from identifying the most dangerous countries to breathe in, to measuring whether current  
> monitoring targets actually enforce the WHO guideline.


In [ ]:
import os, warnings
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
warnings.filterwarnings('ignore')

# ── Path resolution: Kaggle / notebooks subdir / project root ────────────────
KAGGLE = '/kaggle/input/global-environmental-intelligence-dataset/'
if not os.path.exists(KAGGLE):
    # Step up from notebooks/ to project root if needed
    if not os.path.exists('data'):
        os.chdir(os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))

LOCAL = {'who':     'data/raw/who_gho_air_pollution.csv',
         'ihme':    'data/raw/ihme_gbd_air_pollution.csv',
         'daily':   'data/processed/openaq_daily.csv',
         'monthly': 'data/processed/openaq_monthly.csv'}

def load(name):
    if os.path.exists(KAGGLE):
        return pd.read_csv(KAGGLE + os.path.basename(LOCAL[name]))
    return pd.read_csv(LOCAL[name])

who     = load('who')
ihme    = load('ihme')
daily   = load('daily')
monthly = load('monthly')

WHO_LIMIT = 15.0   # µg/m³  WHO 2021 annual PM2.5 guideline
PALETTE   = px.colors.qualitative.Bold

def who_countries(df):
    """Keep only ISO-3 alpha country codes, drop regional aggregates."""
    return df[df['country_code_3'].str.match('^[A-Z]{3}$', na=False)]

print('Data loaded ✓')
for n, d in [('WHO GHO', who), ('IHME GBD', ihme), ('OpenAQ daily', daily)]:
    print(f'  {n}: {len(d):,} rows')

---
## Q1 — Where is the air dirtiest?
### *"Which countries exceed the WHO PM2.5 limit the most, and by how much?"*
> **Business relevance:** Ranking pollution hotspots tells policymakers, investors, and health agencies  
> where to focus limited resources. A country at 9× the WHO limit carries a fundamentally different  
> risk profile than one at 2×.


In [ ]:
pm25 = (who[(who['indicator_code']=='SDGPM25') & (who['sex']=='RESIDENCEAREATYPE_TOTL')]
        .pipe(who_countries)
        .groupby(['country_code_3','year'])['value'].median()
        .reset_index().rename(columns={'value':'pm25'}))

top30 = (pm25.sort_values('year')
             .groupby('country_code_3').last().reset_index()
             .nlargest(30,'pm25').sort_values('pm25'))
top30['multiple'] = (top30['pm25'] / WHO_LIMIT).round(1)
top30['bar_color'] = top30['pm25'].apply(
    lambda v: '#d62728' if v>45 else ('#ff7f0e' if v>30 else '#2ca02c'))

fig = go.Figure(go.Bar(
    y=top30['country_code_3'], x=top30['pm25'], orientation='h',
    marker_color=top30['bar_color'],
    text=top30['pm25'].round(1).astype(str)+' ('+top30['multiple'].astype(str)+'×)',
    textposition='outside',
    hovertemplate='<b>%{y}</b>  PM2.5: %{x:.1f} µg/m³<extra></extra>',
))
fig.add_vline(x=WHO_LIMIT, line_dash='dash', line_color='steelblue', line_width=2,
              annotation_text='WHO limit (15 µg/m³)', annotation_position='top right')
fig.update_layout(
    title='<b>Top 30 Countries by Annual Mean PM2.5 (2023, WHO GHO)</b>',
    xaxis_title='Annual Mean PM2.5 (µg/m³)', height=700,
    template='plotly_white', margin=dict(r=180), font=dict(size=12))
fig.show()

worst = top30.iloc[-1]
exceed_pct = (pm25.groupby('country_code_3')['pm25'].max() > WHO_LIMIT).mean()*100
print(f'Worst: {worst.country_code_3}  {worst.pm25:.1f} µg/m³  ({worst.multiple:.0f}× WHO limit)')
print(f'{exceed_pct:.0f}% of 232 tracked countries exceed the WHO annual PM2.5 guideline')


> **Business Answer:** Qatar, Nigeria, and Egypt lead the 2023 ranking with concentrations  
> 3–4× the WHO limit. Nearly **70% of all tracked countries** exceed 15 µg/m³, confirming air  
> pollution is a global baseline condition, not an outlier. Countries should be classified by  
> "multiples of WHO limit" rather than pass/fail, enabling tiered risk-based policy response.


---
## Q2 — Does more pollution mean more deaths?
### *"Is there a measurable country-level link between PM2.5 concentration and air-pollution mortality?"*
> **Business relevance:** Proves the ROI case for pollution reduction.  
> A strong correlation means each µg/m³ reduced translates to quantifiable lives saved.


In [ ]:
# AIR_10 = WHO mortality proxy (per 100k), 2019
air10 = (who[(who['indicator_code']=='AIR_10') & (who['sex']=='SEX_BTSX')]
         .pipe(who_countries)
         .groupby(['country_code_3','year'])['value'].median()
         .reset_index().rename(columns={'value':'mort_rate'}))

pm25_2019 = pm25[pm25['year']==2019]
q2 = pm25_2019.merge(air10[air10['year']==2019], on=['country_code_3','year'])
q2 = q2[q2['mort_rate'] > 0]

africa = {'NGA','TCD','SLE','MLI','GIN','NER','CIV','CAF','CMR','BFA','GNB','MOZ','MDG',
          'TZA','UGA','ETH','COD','SSD','SDN','ZMB','MWI','BDI','RWA','COG','GHA','SEN',
          'GMB','LBR','BEN','TGO','ERI','DJI','KEN','ZWE','LSO','SWZ','NAM','BWA','ZAF'}
q2['region'] = q2['country_code_3'].apply(
    lambda c: 'Sub-Saharan Africa' if c in africa
    else ('Gulf / MENA' if c in {'QAT','KWT','SAU','ARE','BHR','OMN','EGY','DZA','MAR','LBY'}
    else ('South Asia' if c in {'IND','PAK','BGD','NPL','LKA','AFG'} else 'Rest of World')))

fig = px.scatter(q2, x='pm25', y='mort_rate', color='region', text='country_code_3',
    trendline='ols', trendline_scope='overall',
    color_discrete_sequence=PALETTE,
    labels={'pm25':'Annual PM2.5 (µg/m³)','mort_rate':'Mortality Proxy (WHO, per 100k)','region':'Region'},
    title='<b>PM2.5 vs Air-Pollution Mortality — 183 Countries (2019)</b>',
    template='plotly_white', height=550)
fig.update_traces(textposition='top center', selector=dict(mode='markers+text'))
fig.add_vline(x=WHO_LIMIT, line_dash='dot', line_color='steelblue', annotation_text='WHO limit')
fig.update_layout(font=dict(size=12))
fig.show()

corr = q2[['pm25','mort_rate']].corr().iloc[0,1]
print(f'Pearson r (PM2.5 ↔ mortality): {corr:.2f}')
print('Sub-Saharan Africa: high mortality relative to PM2.5 → indoor biomass burning amplifies burden')
print('Gulf states: extreme PM2.5, moderate mortality → higher incomes / healthcare access cushion')


> **Business Answer:** Correlation is positive (r ≈ 0.6) but not deterministic — two countries  
> with the same PM2.5 can have 10× different mortality rates. Sub-Saharan Africa shows high  
> mortality despite moderate outdoor PM2.5 because **indoor cooking fires** are unmeasured by  
> outdoor monitors. Gulf states have extreme PM2.5 but lower death rates, suggesting **income and  
> healthcare access partially buffer pollution risk.** Pollution reduction ROI therefore differs  
> sharply by region and pollution source.


---
## Q3 — Who bears the highest per-capita death toll?
### *"Which countries have the most air-pollution deaths per 100,000 people?"*
> **Business relevance:** Per-capita rates reveal where intervention has the highest  
> impact-per-dollar. A small country at 200 deaths/100k is easier to shift than a large country at 80.


In [ ]:
# IHME Lelieveld 2019 — country-level death rates
ihme_rates = (ihme.dropna(subset=['death_rates_from_all_air_pollution_per_100000'])
              [['entity','year','death_rates_from_all_air_pollution_per_100000']])
excl = 'World|Asia|Africa|Europe|America|income|SDI|Global|Oceania|East Asia|South Asia|OECD'
ihme_c = ihme_rates[~ihme_rates['entity'].str.contains(excl, case=False, na=False)]

top20 = (ihme_c[ihme_c['year']==2015]
         .nlargest(20,'death_rates_from_all_air_pollution_per_100000')
         .sort_values('death_rates_from_all_air_pollution_per_100000')
         .rename(columns={'death_rates_from_all_air_pollution_per_100000':'rate'}))

fig = go.Figure(go.Bar(
    y=top20['entity'], x=top20['rate'], orientation='h',
    marker=dict(color=top20['rate'], colorscale='Reds', showscale=True,
                colorbar=dict(title='Deaths/100k')),
    text=top20['rate'].round(1), textposition='outside',
    hovertemplate='<b>%{y}</b><br>%{x:.1f} deaths per 100,000<extra></extra>',
))
fig.update_layout(
    title='<b>Top 20 Countries: Air Pollution Death Rate per 100,000 (2015, IHME/Lelieveld)</b>',
    xaxis_title='Age-standardized Deaths per 100,000', height=620,
    template='plotly_white', margin=dict(l=160,r=100), font=dict(size=12))
fig.show()

med = ihme_c[ihme_c['year']==2015]['death_rates_from_all_air_pollution_per_100000'].median()
worst = top20.iloc[-1]
print(f'Worst: {worst.entity}  {worst.rate:.0f}/100k  ({worst.rate/med:.1f}× global median of {med:.0f})')
print('Observation: Eastern Europe dominates — coal/diesel combustion is more lethal than desert dust')


> **Business Answer:** Eastern European countries (Bulgaria, China, North Korea, Hungary)  
> dominate the mortality rate ranking — a striking contrast with Q1, where Africa/Middle East led  
> on raw PM2.5. The explanation: **combustion-generated fine particles** (coal, diesel, wood burning  
> for heat) are more toxic per µg than coarser Saharan dust. Policy targeting combustion sources  
> — EU coal phase-out, building retrofit programmes, vehicle emission standards — will deliver  
> the highest lives-saved per unit of PM2.5 reduced.


---
## Q4 — Are we winning or losing? PM2.5 trends 2010–2023
### *"In the countries with the worst PM2.5, is the situation improving or deteriorating?"*
> **Business relevance:** Trend direction determines urgency. At 0% annual improvement, no country  
> will reach the WHO limit in our lifetimes under current policy.


In [ ]:
top10_iso = pm25[pm25['year']==2023].nlargest(10,'pm25')['country_code_3'].tolist()
trend_df  = pm25[pm25['country_code_3'].isin(top10_iso)].copy()

base = (trend_df[trend_df['year']==2010]
        .set_index('country_code_3')['pm25'].rename('base2010'))
trend_df = trend_df.join(base, on='country_code_3')
trend_df['pct_vs_2010'] = (trend_df['pm25'] - trend_df['base2010']) / trend_df['base2010'] * 100

fig = make_subplots(1,2, subplot_titles=('Absolute PM2.5 (µg/m³)','% Change from 2010'),
                    horizontal_spacing=0.12)
colors = px.colors.qualitative.Plotly
for i, iso in enumerate(top10_iso):
    d = trend_df[trend_df['country_code_3']==iso]
    c = colors[i % len(colors)]
    kw = dict(x=d['year'], line=dict(color=c,width=2), mode='lines+markers',
              marker=dict(size=4), name=iso, legendgroup=iso)
    fig.add_trace(go.Scatter(**kw, y=d['pm25'],
        hovertemplate=f'<b>{iso}</b> %{{x}}: %{{y:.1f}} µg/m³<extra></extra>'), row=1,col=1)
    fig.add_trace(go.Scatter(**kw, y=d['pct_vs_2010'], showlegend=False,
        hovertemplate=f'<b>{iso}</b> %{{x}}: %{{y:+.1f}}%<extra></extra>'), row=1,col=2)

fig.add_hline(y=WHO_LIMIT,row=1,col=1,line_dash='dash',line_color='steelblue',
              annotation_text='WHO 15 µg/m³')
fig.add_hline(y=0, row=1,col=2, line_color='black', line_width=1)
fig.update_yaxes(title_text='PM2.5 (µg/m³)', row=1,col=1)
fig.update_yaxes(title_text='% vs 2010', row=1,col=2)
fig.update_layout(title='<b>PM2.5 Trends — Top 10 Most-Polluted Countries (2010–2023)</b>',
                  height=480, template='plotly_white', font=dict(size=12),
                  legend=dict(x=1.01,y=0.5))
fig.show()

print('CAGR 2010→2023 (negative = improving, positive = worsening):')
for iso in top10_iso:
    d = trend_df[trend_df['country_code_3']==iso].sort_values('year')
    first, last = d.iloc[0]['pm25'], d.iloc[-1]['pm25']
    yrs = d.iloc[-1]['year'] - d.iloc[0]['year']
    if yrs > 0:
        cagr = (last/first)**(1/yrs) - 1
        flag = '🔴' if cagr > 0.005 else ('🟢' if cagr < -0.005 else '🟡')
        print(f'  {flag} {iso}: {cagr*100:+.1f}%/yr  current {last:.1f} µg/m³  ({last/WHO_LIMIT:.1f}× WHO limit)')


> **Business Answer:** Most top-10 polluted countries show **no meaningful improvement or are  
> worsening** over 2010–2023. At these CAGR rates, none will reach the WHO guideline before 2100  
> under business-as-usual. Nepal shows high variability tied to monsoon cycles and seasonal burning —  
> a reminder that annual averages mask dangerous episodic peaks. **Binding 5-year reduction targets  
> backed by sanctions are the only mechanism with a proven track record of sustained improvement.**


---
## Q5 — How effective is the WHO guideline as a real-world constraint?
### *"In 8 countries we directly monitor, how many days exceeded the WHO PM2.5 limit?"*
> **Business relevance:** Monitoring data is the feedback loop for policy. Near-100% exceedance  
> means the guideline is not a target — it's aspirational fiction. 20% exceedance means the limit  
> is achievable with targeted intervention.


In [ ]:
pm25_d = daily[daily['parameter'] == 'pm25'].copy()
pm25_d['exceeds'] = pm25_d['value_ugm3'] > WHO_LIMIT

stats = (pm25_d.groupby('country_code')
         .agg(total=('value_ugm3', 'count'), over=('exceeds', 'sum'),
              peak=('value_ugm3', 'max'), mean=('value_ugm3', 'mean'))
         .assign(pct_over=lambda x: x['over'] / x['total'] * 100)
         .reset_index()
         .sort_values('pct_over', ascending=False))

NAMES = {'BD': 'Bangladesh', 'IN': 'India', 'PK': 'Pakistan', 'EG': 'Egypt',
         'CN': 'China',      'ID': 'Indonesia', 'IL': 'Israel', 'US': 'United States'}
stats['country_name'] = stats['country_code'].map(NAMES)

fig = make_subplots(1, 2,
    subplot_titles=('% of Readings Exceeding WHO PM2.5 Limit (15 µg/m³)',
                    'Mean vs Peak PM2.5 by Country'),
    horizontal_spacing=0.14)

fig.add_trace(go.Bar(
    x=stats['country_name'], y=stats['pct_over'],
    marker_color=['#d62728' if p >= 90 else '#ff7f0e' if p >= 50 else '#2ca02c'
                  for p in stats['pct_over']],
    text=stats['pct_over'].round(0).astype(int).astype(str) + '%',
    textposition='outside',
    hovertemplate='<b>%{x}</b><br>%{y:.1f}% of readings exceed WHO<extra></extra>',
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=stats['mean'], y=stats['peak'],
    mode='markers+text', text=stats['country_name'], textposition='top center',
    marker=dict(size=14, color=stats['pct_over'], colorscale='Reds',
                showscale=True, colorbar=dict(title='%<br>Exceeding')),
    hovertemplate='<b>%{text}</b><br>Mean: %{x:.0f}  Peak: %{y:.0f} µg/m³<extra></extra>',
), row=1, col=2)
fig.add_vline(x=WHO_LIMIT, row=1, col=2, line_dash='dash', line_color='steelblue',
              annotation_text='WHO mean limit')

fig.update_yaxes(title_text='% Readings Over Limit', row=1, col=1)
fig.update_xaxes(title_text='Mean PM2.5 (µg/m³)', row=1, col=2)
fig.update_yaxes(title_text='Peak Single Reading (µg/m³)', row=1, col=2)
fig.update_layout(
    title='<b>WHO PM2.5 Exceedance — 8 Monitored Countries (Dec 2024 – May 2026)</b>',
    height=500, template='plotly_white', font=dict(size=12), showlegend=False)
fig.show()

print(f'{"Country":<16} {"% Over":>8}  {"Mean µg/m³":>10}  {"Peak µg/m³":>10}')
print('-' * 50)
for _, r in stats.iterrows():
    print(f'{r["country_name"]:<16} {r["pct_over"]:>7.1f}%  {r["mean"]:>10.0f}  {r["peak"]:>10.0f}')

> **Business Answer:** Bangladesh, India, Pakistan, and Egypt exceed the WHO limit on  
> **95–100% of measured days** — making the guideline effectively meaningless as a current  
> constraint. Pakistan's single-day peak of **575 µg/m³** is 38× the WHO limit.  
> The United States (17% exceedance) demonstrates that the 15 µg/m³ standard is achievable —  
> but only after decades of sustained Clean Air Act enforcement. **For South Asian governments  
> and their international partners, the lesson is unambiguous: monitoring without enforcement  
> is theatre. Reduction targets must be paired with legal penalties and independent verification.**


---
## Executive Summary

| # | Business Question | Key Finding | Recommended Action |
|---|-------------------|-------------|-------------------|
| 1 | Where is air dirtiest? | 70% of countries exceed WHO limit; Qatar, Nigeria, Egypt lead 2023 rankings | Tiered risk classification by "WHO multiples" |
| 2 | Pollution → deaths? | r ≈ 0.6; Africa's high mortality signals indoor biomass as hidden driver | Separate indoor/outdoor policy tracks by region |
| 3 | Highest per-capita toll? | Eastern Europe leads (coal/diesel); not the most-polluted countries | Target combustion sources, not just concentration |
| 4 | Improving or worsening? | Most top-10 countries show no improvement; current CAGR → WHO target in 100+ years | Mandatory 5-year binding reduction targets |
| 5 | Monitoring → compliance? | BD/IN/PK/EG exceed limit 95–100% of days; only US shows real compliance | Enforcement & penalties, not just monitoring |

---
*Data: WHO GHO · IHME GBD (Lelieveld et al. 2019) · OpenAQ v3 · Open-Meteo ERA5*
